In [1]:
import sys
from pathlib import Path

# 노트북이 juypter-notebook/ 에 있으면 parent 한 번
sys.path.insert(0, str(Path.cwd().parent))
from docling.document_converter import DocumentConverter

# 변환기 생성 (PDF, 이미지 등 여러 포맷 지원)
converter = DocumentConverter()
print("DocumentConverter 준비 완료")

pdf_path = Path("../1_quotation_sample/26818_삼양철강 견적서 2022.05.11.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {pdf_path}")

result = converter.convert(pdf_path)

# display(Markdown(markdown_text))

# doc_dict = result.document.export_to_dict()

# doc_dict

DocumentConverter 준비 완료


In [2]:
# 마크다운을 렌더링해서 이쁘게 보기
from IPython.display import display, Markdown

markdown_text = result.document.export_to_markdown()
display(Markdown(markdown_text))

NO. 202205110001003

## 견적서

## (주)마켓오브메테리얼 貴中

울산광역시 북구 효문동 813-5

(대표) 052-288-8880

윤주선

F a x  :

프로젝트 :

수    신 :

참    조 :

의뢰번호 :

인도조건 :

지불조건 :

납기일자 :

유효기간 :

대표 이사 :

052-288-8880 (직통)

E-mail :

작성자 : 김진우

권 형우 팀장님

T e l  :

VAT포함 :  일금(￦9,276,998)

一金구백이십칠만육천구백구십팔 원정

합계금액 :

페이지 :1

Fax:

2022-05-11

견적 일자 :

<!-- image -->

<!-- image -->

|   순번 NO. | 품명 및 재질 규격 및 메이커   | 길 이 Length   | 수 량 Q'TY   | 단위 UNIT   | 중 량 WEIGHT   | 단위 UNIT   | 단 가 UNIT PRICE   |   금 액 AMOUNT |   비 고 REMARKS |
|------------|-------------------------------|----------------|--------------|-------------|----------------|-------------|--------------------|----------------|-----------------|
|        001 | PIPE SUS304(ERW) 6″ x SCH10S  | 6              | 9            | 본          | 739.8          | 본          | 512,694            |      4,614,246 |                 |
|        002 | PIPE SUS304(ERW) 8″ x SCH10S  | 6              | 1            | 본          | 127.2          | 본          | 740,500            |        740,500 |                 |
|        003 | PIPE SUS304(ERW) 10″ x SCH10S | 6              | 2            | 본          | 314.4          | 본          | 1,030,240          |      2,060,480 |                 |
|        004 | PIPE SUS304(ERW) 16″ x SCH10S | 2              | 1            | 본          | 94.6           | 본          | 375,400            |        375,400 |                 |
|        005 | ㄱ형강 65*65*6 SS275/A36      | 10             | 8            | 본          | 472.8          | KG          | 1,360              |        643,008 |                 |
|            |                               | 이             | 하           | 하          | 여             |             | 백                 |                |                 |
|            | 합 계 / 부 가 세              |                | 21           |             | 1,748.8        |             | 8,433,634          |                |         843,364 |

## 특기사항:

- *4번항 SCH10(6.35T) 생산불가, SCH10S(4.78T) 로 견적 합니다.
- *5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다.

In [3]:
doc_dict = result.document.export_to_dict()
doc_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [4]:
len(doc_dict["groups"])
doc_dict["groups"][1].keys()

dict_keys(['self_ref', 'parent', 'children', 'content_layer', 'name', 'label'])

In [5]:
print(doc_dict["texts"][1].keys())
print()
print(doc_dict["texts"][1])
print(doc_dict["texts"][15])

dict_keys(['self_ref', 'parent', 'children', 'content_layer', 'label', 'prov', 'orig', 'text', 'level'])

{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'section_header', 'prov': [{'page_no': 1, 'bbox': {'l': 245.28, 't': 798.33264, 'r': 330.45768105599996, 'b': 770.25264, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 3]}], 'orig': '견적서', 'text': '견적서', 'level': 1}
{'self_ref': '#/texts/15', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 329.76, 't': 687.12864, 'r': 386.63899200000003, 'b': 677.04864, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 7]}], 'orig': '대표 이사 :', 'text': '대표 이사 :'}


In [6]:
for group in doc_dict["groups"]:
    print(group["name"])
    print(group["self_ref"])

    for d in group["children"]:
        ref = d["$ref"]

        if ref.startswith("#/"):
            ref = ref[2:]
        elif ref.startswith("/"):
            ref = ref[1:]

    cur = doc_dict
    for p in ref.split("/"):
        cur = cur[int(p)] if p.isdigit() else cur[p]
    print(cur["text"])

    print()

group
#/groups/0
합계금액 :

group
#/groups/1
Fax:

group
#/groups/2
견적 일자 :

list
#/groups/3
*5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다.



## 1. Document 타입 분류
* 키워드 매칭을 통해 scoring 하여 분류
* IBM opensource library `DoclingParser` 사용
  * 내부적으로 OCR, PDF loader 모두 지원
  * Vision AI 모델을 내포하고 있음

In [7]:
doc_dict = result.document.export_to_dict()
from document_parsing_engine.app.services import DocumentClassificationService

doc_service = DocumentClassificationService()
classification_result = doc_service.classify(doc_dict)
print("doc_type:", classification_result.doc_type)
print("score:", classification_result.score)
print(
    "reasons:",
    classification_result.reasons[:5] if classification_result.reasons else [],
)

doc_type: DocType.QUOTATION
score: 44
reasons: ['title exact match(section_header): "견적서"', 'text contains "견적": "견적서"', 'text contains "견적": "견적 일자 :"', 'text contains "견적": "*4번항 SCH10(6.35T) 생산불가, SCH10S(4.78T) 로 견적 합니다."', 'text contains "견적": "*5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다."']


## 2. Document Layout Parsing
* doclingParser는 문서를 영역 별로 읽음
  * 문서의 유사한 정보는 __한 곳에 모여 있고 근처에 있다고 판단함__
  * 모여있는 텍스트를 하나의 block으로 인식
  *  block 기본적으로 단순 `text`, `group`, `table`, `picture` 로 나뉨
    * group은 `list`, `key-value` 로 나뉨

In [8]:
from document_parsing_engine.app.services import DocumentLayoutParsingService

svc = DocumentLayoutParsingService(y_tol=10, min_gap_threshold=18, gap_ratio=1.8)
# blocks_sorted = svc.get_blocks_sorted(doc_dict)

# # 그룹 ref별 known_keys (선택). 비우면 기본 파싱만 사용
# group_known_keys_map = {}  # 예: {"#/groups/0": {"견적 일자", "수신", "합계금액"}}

# # 구조화된 결과만 필요할 때
# blocks = svc.parse_blocks(
#     doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
# )
# for blk in blocks:
#     print(blk.ref, blk.content_type, blk.content)  # 노트북처럼 출력만 할 때
# svc.print_blocks_content(
#     doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
# )

In [9]:
blocks_sorted = svc.get_blocks_sorted(doc_dict)

# 그룹 ref별 known_keys (선택). 비우면 기본 파싱만 사용
group_known_keys_map = {}  # 예: {"#/groups/0": {"견적 일자", "수신", "합계금액"}}

# 구조화된 결과만 필요할 때
blocks = svc.parse_blocks(doc_dict)

# 노트북처럼 출력만 할 때
svc.print_blocks_content(
    doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
)


[0] #/texts/0  label=text
NO. 202205110001003

[1] #/texts/1  label=section_header
견적서

[2] #/texts/2  label=section_header
(주)마켓오브메테리얼 貴中

[3] #/groups/0  label=key_value_area
    - 수 신: 권 형우 팀장님
    - 참 조: 
    - 프로젝트: 
    - 대표 이사: 윤주선
    - 의뢰번호: 울산광역시 북구 효문동 813-5
    - 인도조건: 
    - T e l: (대표) 052-288-8880
    - 지불조건: 
    - F a x: 052-288-8880 (직통)
    - 납기일자: 작성자 : 김진우
    - 유효기간: E-mail :
    - VAT포함: 일금(￦9,276,998)
    - 합계금액: 一金구백이십칠만육천구백구십팔 원정

[4] #/groups/1  label=key_value_area
    - Tel: 
    - Fax: 
    - 페이지: 1

[5] #/groups/2  label=key_value_area
    - 견적 일자: 2022-05-11

[6] #/pictures/0  label=picture
    [PICTURE]

[7] #/pictures/1  label=picture
    [PICTURE]

[8] #/tables/0  label=table
    row[0] = ['순번 NO.', '품명 및 재질 규격 및 메이커', '길 이 Length', "수 량 Q'TY", '단위 UNIT', '중 량 WEIGHT', '단위 UNIT', '단 가 UNIT PRICE', '금 액 AMOUNT', '비 고 REMARKS']
    row[1] = ['001', 'PIPE SUS304(ERW) 6″ x SCH10S', '6', '9', '본', '739.8', '본', '512,694', '4,614,246', '']
    row[2] = ['0

In [10]:
# svc.parse_blocks(doc_dict)

## 3. Segment Layout mapping
* 견적서는 `quotation_meta`, `items`, `remarks` 같은 세그먼트로 나뉨
* 위에서 parsing 된 블럭이 어떤 세그먼트인지 매핑해야함
* LLM이 필드 해석을 하기 위한 깨끗한 정보를 주기 위해서임
  
  * `ex prompt`
    * quotation_meta 란 철강 견적서에서 제공자, 고객, 일자 등이 포함된 세그먼트야.
    * quotation_meta 세그먼트에 추측되는 정보들로 아래와 같은 같은 정보들이 있어. 여기서 ~~ 데이터를 뽑아줘
    * RAG 활용시 - 철강 도메인지식 문서 제공

``` bash
quotation_meta
├── document_key
├── supplier_name
├── supplier_contact_person
├── supplier_phone
├── supplier_address
├── buyer_name
├── buyer_contact_person
├── buyer_phone
├── buyer_address
├── validity # (견적 유효기간)
├── payment_terms # (대금지급 조건)
└── delivery_terms # (운송 조건)

items
├── rows[]
│   ├── unit_price # (단가)
│   ├── quantity 
│   ├── line_amount # (단가 X 수량) 특정 항목에 대한 순수 금액 - 부가세, 배송비, 할인 제외)
│   ├── vat
│   ├── material_grade # (재질)
│   ├── thickness # (두께)
│   ├── size_text # (사이즈)
│   └── spec_text
└── totals
    ├── supply_amount
    └── total_amount
    
remarks
├── raw_text
└── extracted_conditions(optional)
```

In [11]:
from document_parsing_engine.app.services.layout_segment_mapping_service import (
    LayoutSegmentMappingService,
)

mapper = LayoutSegmentMappingService()
seg = mapper.recommend(
    doc_type=classification_result.doc_type,
    doc_dict={},
    blocks=blocks,
)
mapper.pretty_print(seg, blocks)

=== segment: quotation_meta ===
  block_refs: ['#/texts/1', '#/groups/0', '#/groups/1']
  reasons: ['segment keywords matched: 견적서', 'title keywords matched: 견적서', 'content_type=text']...
    - #/texts/1 | label=section_header | content_type=text
      견적서
    - #/groups/0 | label=key_value_area | content_type=group_kv
      수 신: 권 형우 팀장님
      참 조: 
      프로젝트: 
      대표 이사: 윤주선
      의뢰번호: 울산광역시 북구 효문동 813-5
      인도조건: 
      T e l: (대표) 052-288-8880
      지불조건: 
      F a x: 052-288-8880 (직통)
      납기일자: 작성자 : 김진우
      유효기간: E-mail :
      VAT포함: 일금(￦9,276,998)
      합계금액: 一金구백이십칠만육천구백구십팔 원정
    - #/groups/1 | label=key_value_area | content_type=group_kv
      Tel: 
      Fax: 
      페이지: 1

=== segment: items ===
  block_refs: ['#/tables/0']
  reasons: ["field keywords matched: unit price, price, q'ty, amount, 재질, length, 규격, 메이커", 'segment keywords matched: 품명, 재질, 규격', 'content_type=table']...
    - #/tables/0 | label=table | content_type=table
       ['순번 NO.', '품명 및 재질 규격 및 메

## 4. Segment Reconstruction
* 견적서는 `quotation_meta`, `items`, `remarks` 같은 세그먼트로 나뉨
* 아래와 같은 경우 segment 를 재구성 할 필요가 있음
  * 보통 quotation_meta는 `key-value` 형태 (T e l: (대표) 052-288-8880) 와 같은 형태이나 `table(표)` 로 된 경우
  * 반대로 items는 `table(표)` 형태이나 `key-value` 형태나 표 구조가 뭉개져 `text` 형태로 나눠져 있는 경우
* quotation_meta : table-> key-value
* items          : key-value, text -> table

In [12]:
### 개발 필요

## 5. Field Mapping
* 소넷 사용

In [13]:
# print(QUOTATION_SEGMENT_DEFINITION.doc_type)
# print(QUOTATION_SEGMENT_DEFINITION.aliases)
# for s in QUOTATION_SEGMENT_DEFINITION.segments:
#     print(s.name)
#     print(s.description)
#     print("+++++++++++++++")
#     print("+++++++++++++++", s.name)

#     for f in s.fields:
#         print("=========", f.name)
#         print(f.description)
#         print(f.keyword_def)
#         print(f.keyword_def.keywords)
#         print(f.keyword_def.boost_score)
#         print()


seg

LayoutSegmentMappingRecommendation(doc_type='quotation', allowed_segments=['quotation_meta', 'items', 'remarks'], block_recommendations=[BlockSegmentRecommendation(block_ref='#/texts/0', label='text', content_type='text', primary_segment=None, candidates=[SegmentCandidate(segment_name='quotation_meta', score=0.47500000000000003, reasons=['field keywords matched: no.', 'content_type=text', 'label=text'])]), BlockSegmentRecommendation(block_ref='#/texts/1', label='section_header', content_type='text', primary_segment='quotation_meta', candidates=[SegmentCandidate(segment_name='quotation_meta', score=0.72, reasons=['segment keywords matched: 견적서', 'title keywords matched: 견적서', 'content_type=text', 'label=section_header'])]), BlockSegmentRecommendation(block_ref='#/texts/2', label='section_header', content_type='text', primary_segment=None, candidates=[SegmentCandidate(segment_name='quotation_meta', score=0.445, reasons=['field keywords matched: 貴中', 'content_type=text', 'label=section_he

In [14]:
# LayoutSegmentMappingRecommendation에서 세그먼트별 데이터 추출
def get_segment_data(recommendation, blocks):
    """segment_buckets 기준으로 세그먼트별 블록 데이터를 dict로 반환. { segment_name: [block, ...] }"""
    blocks_by_ref = {
        getattr(b, "ref", None): b for b in blocks if getattr(b, "ref", None)
    }
    segment_data = {}
    for bucket in recommendation.segment_buckets:
        segment_data[bucket.segment_name] = [
            blocks_by_ref[ref] for ref in bucket.block_refs if ref in blocks_by_ref
        ]
    # 미매핑 블록도 함께 반환 (선택)
    segment_data["_unmapped"] = [
        blocks_by_ref[ref]
        for ref in recommendation.unmapped_block_refs
        if ref in blocks_by_ref
    ]
    return segment_data


segment_data = get_segment_data(seg, blocks)

for name, blks in segment_data.items():
    print(f"=== {name} ({len(blks)} blocks) ===")
    for b in blks:
        ct = getattr(b, "content_type", "")
        content = getattr(b, "content", "")
        if ct == "text":
            print(f"  {b.ref}:")
            print(f"    {content}")
        elif ct == "group_kv" and isinstance(content, dict):
            print(f"  {b.ref}:")
            # print(type(content))
            # for i in content.items():
            print(f"    ", content)
        elif ct == "table" and isinstance(content, list):
            print(f"  {b.ref}: (rows={len(content)})")
            for i, row in enumerate(content):
                print(f"    {row}")
        else:
            print(f"  {b.ref}: ({ct}) {content}")
    print()

=== quotation_meta (3 blocks) ===
  #/texts/1:
    견적서
  #/groups/0:
     {'수 신': '권 형우 팀장님', '참 조': '', '프로젝트': '', '대표 이사': '윤주선', '의뢰번호': '울산광역시 북구 효문동 813-5', '인도조건': '', 'T e l': '(대표) 052-288-8880', '지불조건': '', 'F a x': '052-288-8880 (직통)', '납기일자': '작성자 : 김진우', '유효기간': 'E-mail :', 'VAT포함': '일금(￦9,276,998)', '합계금액': '一金구백이십칠만육천구백구십팔 원정'}
  #/groups/1:
     {'Tel': '', 'Fax': '', '페이지': '1'}

=== items (1 blocks) ===
  #/tables/0: (rows=8)
    ['순번 NO.', '품명 및 재질 규격 및 메이커', '길 이 Length', "수 량 Q'TY", '단위 UNIT', '중 량 WEIGHT', '단위 UNIT', '단 가 UNIT PRICE', '금 액 AMOUNT', '비 고 REMARKS']
    ['001', 'PIPE SUS304(ERW) 6″ x SCH10S', '6', '9', '본', '739.8', '본', '512,694', '4,614,246', '']
    ['002', 'PIPE SUS304(ERW) 8″ x SCH10S', '6', '1', '본', '127.2', '본', '740,500', '740,500', '']
    ['003', 'PIPE SUS304(ERW) 10″ x SCH10S', '6', '2', '본', '314.4', '본', '1,030,240', '2,060,480', '']
    ['004', 'PIPE SUS304(ERW) 16″ x SCH10S', '2', '1', '본', '94.6', '본', '375,400', '375,400', '']
    [

In [ ]:
from document_parsing_engine.domain.presets import QUOTATION_SEGMENT_DEFINITION

for s in QUOTATION_SEGMENT_DEFINITION.segments:

    print("+++++++++++++++")
    print("+++++++++++++++", s.name)

    for f in s.fields:
        print("====", f.name)
        print("====", f.description)
        print("====", f.keyword_def.keywords)
        print()
        if f.children:
            for c in f.children:
                print("=========", c.name)
                print("=========", c.description)
                print("=========", c.keyword_def.keywords)
                print()
    print()
    print()

+++++++++++++++
+++++++++++++++ quotation_meta
==== document_key
==== 문서 식별 키
==== ('견적번호', 'no.', '문서번호', 'document no')

==== supplier_name
==== 공급업체 상호
==== ('공급업체', 'supplier', '판매처', '상호', '회사명')

==== supplier_contact_person
==== 공급업체 담당자명
==== ('담당자', '작성자', 'contact person')

==== supplier_phone
==== 공급업체 연락처
==== ('tel', '전화', '연락처', 'phone', 'fax', 'email', 'e-mail')

==== supplier_address
==== 공급업체 주소
==== ('주소', 'address')

==== buyer_name
==== 고객사 상호
==== ('수신', '귀중', '貴中', 'buyer', '고객사')

==== buyer_contact_person
==== 고객사 담당자명
==== ('수신', '참조', '담당자')

==== buyer_phone
==== 고객사 연락처
==== ('tel', '전화', '연락처')

==== buyer_address
==== 고객사 주소
==== ('주소', 'address')

==== validity
==== 견적 유효기간
==== ('유효기간', 'validity')

==== payment_terms
==== 대금지급 조건
==== ('지불조건', '결제조건', 'payment')

==== delivery_terms
==== 운송 조건
==== ('인도조건', '운송조건', 'delivery', '납기일자')



+++++++++++++++
+++++++++++++++ items
==== rows
==== 품목 행 목록
==== ()

========= unit_price
========= 단가
========= ('단

In [ ]:
from document_parsing_engine.domain.presets.segment_definitions import (
    get_segment_definition,
)

seg_def = get_segment_definition(classification_result.doc_type.value)  # 'quotation'
seg_def.doc_type
# get_segment_definition("purchase_order").doc_type  # 'purchase_order'

'quotation'

In [31]:
# QUOTATION_SEGMENT_DEFINITION.segments → 세그먼트별 필드 키워드 매핑 { segment_name: { field_name: [keywords, ...] } }
from document_parsing_engine.domain.presets.segment_definitions import (
    QUOTATION_SEGMENT_DEFINITION,
)


def build_segment_field_keywords(segments):
    """세그먼트 정의에서 세그먼트별·필드별 키워드 매핑. 자식 없음: field -> [keywords]; 자식 있음: field -> {"keywords": [...], child_name: [...]}."""
    out = {}
    for seg in segments:
        out[seg.name] = {}
        for f in seg.fields:
            kw = list(getattr(f.keyword_def, "keywords", ()) or ())
            if not f.children:
                out[seg.name][f.name] = kw
            else:
                out[seg.name][f.name] = {"keywords": kw}
                for c in f.children:
                    out[seg.name][f.name][c.name] = list(
                        getattr(c.keyword_def, "keywords", ()) or ()
                    )
    return out


segment_field_keywords = build_segment_field_keywords(seg_def.segments)
segment_field_keywords

{'quotation_meta': {'document_key': ['견적번호', 'no.', '문서번호', 'document no'],
  'supplier_name': ['공급업체', 'supplier', '판매처', '상호', '회사명'],
  'supplier_contact_person': ['담당자', '작성자', 'contact person'],
  'supplier_phone': ['tel', '전화', '연락처', 'phone', 'fax', 'email', 'e-mail'],
  'supplier_address': ['주소', 'address'],
  'buyer_name': ['수신', '귀중', '貴中', 'buyer', '고객사'],
  'buyer_contact_person': ['수신', '참조', '담당자'],
  'buyer_phone': ['tel', '전화', '연락처'],
  'buyer_address': ['주소', 'address'],
  'validity': ['유효기간', 'validity'],
  'payment_terms': ['지불조건', '결제조건', 'payment'],
  'delivery_terms': ['인도조건', '운송조건', 'delivery', '납기일자']},
 'items': {'rows': {'keywords': [],
   'unit_price': ['단가', 'unit price', 'price'],
   'quantity': ['수량', 'qty', "q'ty", 'quantity'],
   'line_amount': ['금액', 'amount'],
   'material_grade': ['재질', 'grade', 'material', 'description'],
   'thickness': ['두께', 'thickness'],
   'size_text': ['사이즈', 'size', '길이', '폭', 'length'],
   'spec_text': ['규격', 'spec', 'maker

In [ ]:
classification_result.doc_type.value

'quotation'

In [30]:
segment_data

{'quotation_meta': [BlockContent(ref='#/texts/1', label='section_header', content_type='text', content='견적서'),
  BlockContent(ref='#/groups/0', label='key_value_area', content_type='group_kv', content={'수 신': '권 형우 팀장님', '참 조': '', '프로젝트': '', '대표 이사': '윤주선', '의뢰번호': '울산광역시 북구 효문동 813-5', '인도조건': '', 'T e l': '(대표) 052-288-8880', '지불조건': '', 'F a x': '052-288-8880 (직통)', '납기일자': '작성자 : 김진우', '유효기간': 'E-mail :', 'VAT포함': '일금(￦9,276,998)', '합계금액': '一金구백이십칠만육천구백구십팔 원정'}),
  BlockContent(ref='#/groups/1', label='key_value_area', content_type='group_kv', content={'Tel': '', 'Fax': '', '페이지': '1'})],
 'items': [BlockContent(ref='#/tables/0', label='table', content_type='table', content=[['순번 NO.', '품명 및 재질 규격 및 메이커', '길 이 Length', "수 량 Q'TY", '단위 UNIT', '중 량 WEIGHT', '단위 UNIT', '단 가 UNIT PRICE', '금 액 AMOUNT', '비 고 REMARKS'], ['001', 'PIPE SUS304(ERW) 6″ x SCH10S', '6', '9', '본', '739.8', '본', '512,694', '4,614,246', ''], ['002', 'PIPE SUS304(ERW) 8″ x SCH10S', '6', '1', '본', '127.2', '본', '7

In [ ]:
# segment_data → JSON (BlockContent를 dict로 변환 후 직렬화)
import json


def segment_data_to_json_ready(segment_data):
    """segment_data(세그먼트별 BlockContent 리스트)를 json.dumps 가능한 dict로 변환."""
    out = {}
    for name, blocks in segment_data.items():
        out[name] = []
        for b in blocks:
            out[name].append(
                {
                    "ref": getattr(b, "ref", None),
                    "label": getattr(b, "label", None),
                    "content_type": getattr(b, "content_type", None),
                    "content": getattr(b, "content", None),
                }
            )
    return out


segment_data_json = segment_data_to_json_ready(segment_data)
segment_data_json_str = json.dumps(segment_data_json, ensure_ascii=False, indent=2)
# 파일 저장: Path("segment_data.json").write_text(segment_data_json_str, encoding="utf-8")
segment_data_json

{'quotation_meta': [{'ref': '#/texts/1',
   'label': 'section_header',
   'content_type': 'text',
   'content': '견적서'},
  {'ref': '#/groups/0',
   'label': 'key_value_area',
   'content_type': 'group_kv',
   'content': {'수 신': '권 형우 팀장님',
    '참 조': '',
    '프로젝트': '',
    '대표 이사': '윤주선',
    '의뢰번호': '울산광역시 북구 효문동 813-5',
    '인도조건': '',
    'T e l': '(대표) 052-288-8880',
    '지불조건': '',
    'F a x': '052-288-8880 (직통)',
    '납기일자': '작성자 : 김진우',
    '유효기간': 'E-mail :',
    'VAT포함': '일금(￦9,276,998)',
    '합계금액': '一金구백이십칠만육천구백구십팔 원정'}},
  {'ref': '#/groups/1',
   'label': 'key_value_area',
   'content_type': 'group_kv',
   'content': {'Tel': '', 'Fax': '', '페이지': '1'}}],
 'items': [{'ref': '#/tables/0',
   'label': 'table',
   'content_type': 'table',
   'content': [['순번 NO.',
     '품명 및 재질 규격 및 메이커',
     '길 이 Length',
     "수 량 Q'TY",
     '단위 UNIT',
     '중 량 WEIGHT',
     '단위 UNIT',
     '단 가 UNIT PRICE',
     '금 액 AMOUNT',
     '비 고 REMARKS'],
    ['001',
     'PIPE SUS304(ERW) 6″ x 

In [ ]:
from anthropic import Anthropic
import time
from typing import Any

import os
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
client = Anthropic(api_key=api_key)


model_mapping = {"claude-sonnet": "claude-sonnet-4-6"}
actual_model = model_mapping.get("claude-sonnet")

start_time = time.time()
message = client.messages.create(
    model=actual_model,
    max_tokens=50,
    messages=[{"role": "user", "content": "안녕하세요! 간단한 연결 테스트입니다."}],
)
response_time = time.time() - start_time

test_response_text = (
    message.content[0].text[:100] + "..."
    if message.content
    and len(message.content) > 0
    and len(message.content[0].text) > 100
    else (
        message.content[0].text if message.content and len(message.content) > 0 else ""
    )
)

test_response_text

'안녕하세요! 👋\n\n연결이 잘 되고 있네요! 😊\n\n무엇을 도와드릴까요?'

In [ ]:
# 프롬프트 넣어서 호출 (위쪽 셀에서 client, actual_model 사용)
def call_with_prompt(
    prompt: str,
    model: str | None = None,
    max_tokens: int = 1024,
    client_instance=None,
) -> str:
    """프롬프트 문자열을 넣으면 Claude API를 호출하고 응답 텍스트를 반환."""
    c = client_instance or client
    m = model or actual_model
    msg = c.messages.create(
        model=m,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    if msg.content and len(msg.content) > 0:
        return msg.content[0].text
    return ""


# extraction_prompts.yaml 에서 시스템 + 유저 프롬프트 로드 → 합쳐서 출력 후 API 호출
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from document_parsing_engine.prompt import get_extraction_prompts

# 변수: segment_field_mapping_metadata, segment_mapping_result_data
try:
    _ = segment_data_json_str
except NameError:
    _sd = segment_data if "segment_data" in dir() else {}
    _json = {
        n: [
            {
                "ref": getattr(b, "ref", None),
                "label": getattr(b, "label", None),
                "content_type": getattr(b, "content_type", None),
                "content": getattr(b, "content", None),
            }
            for b in blks
        ]
        for n, blks in _sd.items()
    }
    segment_data_json_str = json.dumps(_json, ensure_ascii=False, indent=2)

variables = {
    "segment_field_mapping_metadata": (
        json.dumps(segment_field_keywords, ensure_ascii=False, indent=2)
        if "segment_field_keywords" in dir()
        else "{}"
    ),
    "segment_mapping_result_data": (
        segment_data_json_str
        if "segment_data_json_str" in dir()
        else json.dumps({}, ensure_ascii=False)
    ),
}
system_prompt, user_prompt = get_extraction_prompts(variables)

# 시스템 + 유저 프롬프트 합쳐서 출력
combined = (
    "=== 시스템 프롬프트 ===\n\n"
    + system_prompt
    + "\n\n=== 유저 프롬프트 ===\n\n"
    + user_prompt
)
print(combined)

=== 시스템 프롬프트 ===

당신은 철강 산업 전문 문서 세그먼트-필드 매핑 전용 정보추출기이다.

목표:
입력으로 주어진
1) 세그먼트-필드 매핑 정보 메타 데이터
2) 문서에서 추출한 세그먼트-필드 매핑 결과 데이터
를 바탕으로,
지정된 출력 템플릿 구조에 맞는 JSON만 생성한다.

반드시 지켜야 할 규칙:

[기본 원칙]
1. 세그먼트는 우선 동일한 세그먼트끼리 비교한다.
   - quotation_meta ↔ quotation_meta
   - items ↔ items
   - remarks ↔ remarks
2. 세그먼트-필드 매핑 정보는 다음 구조를 가진다.
   - {세그먼트: {필드: 필드로 인식되는 키워드 리스트}}
3. 출력은 반드시 지정된 템플릿과 동일한 JSON 구조여야 한다.
4. 100% 매핑되지 않을 수 있으므로, 확신이 없으면 빈 값 또는 null을 유지한다.
5. 억지 매핑을 하지 않는다.
6. 설명문, 주석, 마크다운 없이 JSON만 출력한다.

[세그먼트별 처리 규칙]

A. quotation_meta 처리
1. 문서 추출 데이터의 quotation_meta 세그먼트에서
   - label == "key_value_area"
   - content_type == "group_kv"
   인 요소를 우선 사용한다.
2. 각 content의 key를 보고,
   세그먼트-필드 매핑 정보의 quotation_meta 각 필드에 정의된 키워드 리스트와
   가장 유사한 key를 해당 필드로 매핑한다.
3. value를 최종 필드값으로 사용한다.
4. 하나의 value 안에 다른 정보가 섞여 있을 수 있다.
   예:
   - "작성자 : 김진우" → supplier_contact_person 후보
   - "E-mail :" → 값이 비어있으면 공란 처리
5. quotation_meta에서 부족한 필드는 _unmapped 세그먼트에서 보완할 수 있다.
   특히 다음을 우선 고려한다.
   - content_type == "te

In [ ]:
# (선택) API 호출: system + user 로 전달
msg = client.messages.create(
    model=actual_model,
    max_tokens=4096,
    system=system_prompt,
    messages=[{"role": "user", "content": user_prompt}],
)
response_text = msg.content[0].text if msg.content else ""
print("\n=== API 응답 ===\n")
print(response_text)
# response_text


=== API 응답 ===

```json
{
  "quotation_meta": {
    "document_key": "NO. 202205110001003",
    "supplier_name": "",
    "supplier_contact_person": "김진우",
    "supplier_phone": "(대표) 052-288-8880",
    "supplier_address": "",
    "buyer_name": "(주)마켓오브메테리얼",
    "buyer_contact_person": "권 형우",
    "buyer_phone": "",
    "buyer_address": "",
    "validity": "",
    "payment_terms": "",
    "delivery_terms": ""
  },
  "items": {
    "rows": [
      {
        "unit_price": 512694,
        "quantity": 9,
        "line_amount": 4614246,
        "vat": null,
        "material_grade": "SUS304",
        "thickness": "SCH10S",
        "size_text": "6″",
        "spec_text": "PIPE SUS304(ERW) 6″ x SCH10S"
      },
      {
        "unit_price": 740500,
        "quantity": 1,
        "line_amount": 740500,
        "vat": null,
        "material_grade": "SUS304",
        "thickness": "SCH10S",
        "size_text": "8″",
        "spec_text": "PIPE SUS304(ERW) 8″ x SCH10S"
      },
      {
        "u

'```json\n{\n  "quotation_meta": {\n    "document_key": "NO. 202205110001003",\n    "supplier_name": "",\n    "supplier_contact_person": "김진우",\n    "supplier_phone": "(대표) 052-288-8880",\n    "supplier_address": "",\n    "buyer_name": "(주)마켓오브메테리얼",\n    "buyer_contact_person": "권 형우",\n    "buyer_phone": "",\n    "buyer_address": "",\n    "validity": "",\n    "payment_terms": "",\n    "delivery_terms": ""\n  },\n  "items": {\n    "rows": [\n      {\n        "unit_price": 512694,\n        "quantity": 9,\n        "line_amount": 4614246,\n        "vat": null,\n        "material_grade": "SUS304",\n        "thickness": "SCH10S",\n        "size_text": "6″",\n        "spec_text": "PIPE SUS304(ERW) 6″ x SCH10S"\n      },\n      {\n        "unit_price": 740500,\n        "quantity": 1,\n        "line_amount": 740500,\n        "vat": null,\n        "material_grade": "SUS304",\n        "thickness": "SCH10S",\n        "size_text": "8″",\n        "spec_text": "PIPE SUS304(ERW) 8″ x SCH10S"\n      

In [45]:
print(response_text)

```json
{
  "quotation_meta": {
    "document_key": "NO. 202205110001003",
    "supplier_name": "",
    "supplier_contact_person": "김진우",
    "supplier_phone": "(대표) 052-288-8880",
    "supplier_address": "",
    "buyer_name": "(주)마켓오브메테리얼",
    "buyer_contact_person": "권 형우",
    "buyer_phone": "",
    "buyer_address": "",
    "validity": "",
    "payment_terms": "",
    "delivery_terms": ""
  },
  "items": {
    "rows": [
      {
        "unit_price": 512694,
        "quantity": 9,
        "line_amount": 4614246,
        "vat": null,
        "material_grade": "SUS304",
        "thickness": "SCH10S",
        "size_text": "6″",
        "spec_text": "PIPE SUS304(ERW) 6″ x SCH10S"
      },
      {
        "unit_price": 740500,
        "quantity": 1,
        "line_amount": 740500,
        "vat": null,
        "material_grade": "SUS304",
        "thickness": "SCH10S",
        "size_text": "8″",
        "spec_text": "PIPE SUS304(ERW) 8″ x SCH10S"
      },
      {
        "unit_price": 10302